# Step 0: The big picture

An LLM does **one thing**: given some tokens, it **guesses the next token**. Chatting, coding, answering questions: all of it is this one guess, repeated in a loop.

```
text → tokenizer → IDs → embeddings (+ positions) → transformer blocks → scores → softmax → probabilities → pick → append → repeat
```

In this notebook we build the **simplest possible next-word guesser**: a counting model. It already runs the whole loop (guess, pick, append, repeat). Then we see exactly why it is not enough. That is why the real LLM parts (attention, positions, …) exist.

**How to use this notebook**
- Run the cells from top to bottom (Shift+Enter).
- In **Your turn** cells, replace `None` with your answer and run the cell. `check(...)` tells you if you are right, and gives a hint if not.
- Stuck? Run `solution("0.1")` (with the exercise number).

In [ ]:
# Make the `toddler` package (repo root) importable from this notebook
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

from toddler.data import CORPUS
from toddler.exercises import check, solution

## 0.1 The toddler's whole world

Everything the toddler has ever heard. For now we cut the text at spaces; Step 1 replaces this with a real tokenizer (BPE).

In [ ]:
for sentence in CORPUS:
    print(sentence)

words = [word for sentence in CORPUS for word in sentence.split()]
print(f"\n{len(words)} words heard, {len(set(words))} different: {sorted(set(words))}")

## 0.2 The guessing game

Training means: hide the next word, guess it, check. Each 8-word sentence gives 7 guesses.

In [ ]:
games = []
for sentence in CORPUS:
    sentence_words = sentence.split()
    for i in range(1, len(sentence_words)):
        games.append((" ".join(sentence_words[:i]), sentence_words[i]))

for seen, target in games:
    print(f"{seen:<28} → must guess: {target}")
print(f"\n{len(games)} guesses in total")

## 0.3 The simplest model: count what comes next

Look **only at the last word**, and count what came after it in the training data. This is called a *bigram* model. Its "weights" are just these counts.

In [ ]:
from collections import Counter, defaultdict

def count_next_words(corpus):
    counts = defaultdict(Counter)
    for sentence in corpus:
        sentence_words = sentence.split()
        for current, following in zip(sentence_words, sentence_words[1:]):
            counts[current][following] += 1
    return counts

next_counts = count_next_words(CORPUS)
for word, counter in next_counts.items():
    print(f"after {word!r:<9} heard: {dict(counter)}")

To turn counts into **probabilities**, divide each count by the total for that word. Example: after `am` the toddler heard `hungry` 1 time out of 3, so P(hungry | am) = 1/3.

### Your turn 0.1
Using the counts above, what are the probabilities of the next word after `I`? Write numbers like `0.25`, not `25%`.

In [ ]:
my_answer = {"am": None, "want": None}
check("0.1", my_answer)

In [ ]:
def next_word_probabilities(counts, word):
    total = sum(counts[word].values())
    return {following: count / total for following, count in counts[word].items()}

for word in next_counts:
    probabilities = next_word_probabilities(next_counts, word)
    print(f"after {word!r:<9} → " + ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items()))

## 0.4 Generating: guess, pick, append, repeat

This loop is exactly how ChatGPT writes, one token at a time. We **pick at random, following the probabilities**: a 50% word gets picked about half of the time.

In [ ]:
import random

def generate(counts, start="I", length=8, seed=None, verbose=False):
    rng = random.Random(seed)
    words = [start]
    while len(words) < length:
        probabilities = next_word_probabilities(counts, words[-1])
        picked = rng.choices(list(probabilities), weights=list(probabilities.values()))[0]
        if verbose:
            options = ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items())
            print(f"{' '.join(words):<26} | last word {words[-1]!r:<8} → {options:<30} → picked {picked!r}")
        words.append(picked)
    return " ".join(words)

print("\nresult:", generate(next_counts, seed=1, verbose=True))

In [ ]:
for seed in range(8):
    print(generate(next_counts, seed=seed))

## 0.5 Why counting is not enough

Look at the sentences above. Two problems:

1. After `want`, the model only sees `want`, so it gives milk, mommy and bed 1/3 each. It **cannot look back** at `hungry`. Fix: **attention** (Step 4).
2. After `I`, it says am 50% / want 50%. It **cannot tell** the first `I` from the second. Fix: **context and positions** (Steps 3 and 4).

### Your turn 0.2
After `I am hungry . I want`, what probability does the counting model give to `milk`? And what would a perfect model give?

In [ ]:
my_answer = {"counting model": None, "perfect model": None}
check("0.2", my_answer)

### Your turn 0.3 (challenge)
What fraction of generated 8-word sentences are exactly one of the 3 training sentences? Work it out with probabilities first, then run the cell after it to measure.

In [ ]:
my_answer = None  # a number between 0 and 1
check("0.3", my_answer)

In [ ]:
training_sentences = set(CORPUS)
samples = [generate(next_counts, seed=seed) for seed in range(10_000)]
share = sum(sample in training_sentences for sample in samples) / len(samples)
print(f"{share:.1%} of 10,000 generated sentences are training sentences")

### Experiment
The model's probabilities simply mirror how often it heard things. Add sentences (you can even invent new words), re-run, and watch the probabilities change.

In [ ]:
my_corpus = CORPUS + [
    "I am hungry . I want milk .",
]
my_counts = count_next_words(my_corpus)
for word in ["I", "am", "want"]:
    probabilities = next_word_probabilities(my_counts, word)
    print(f"after {word!r:<6} → " + ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items()))
print()
for seed in range(5):
    print(generate(my_counts, seed=seed))

## Summary

- An LLM only **guesses the next token**; writing is guess → pick → append → repeat.
- It outputs **probabilities**, not one answer. When the text gives no clue (after `am`), the best answer is to share the probability.
- Its knowledge lives in its **weights**. Here, the weights are the counts table.
- Counting fails because it only looks at **the last word**. The transformer fixes this by looking back at the whole context.

Next: **Step 1, tokenization** (`01_tokenization_bpe.ipynb`).